# EN4554 Assignment 2 — Convolutional Neural Networks

**Name:** Rivikula Fernando
**Index number:** _<your index>_

**AI tool use:** Claude (Anthropic) was used to help write and debug parts of the code in this notebook.
All code was read, run, and checked by me, and the results and interpretations are my own.

Runtime: Google Colab, NVIDIA T4 GPU, PyTorch.

## Setup

In [ ]:
import random, time, math, json, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'no GPU')

SEED = 0
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
torch.backends.cudnn.benchmark = True  # faster convolutions; runs are seeded but not bit-exact

### Data

CIFAR-10 training set (50k) is split into **45k train / 5k validation** with a fixed seed.
Per-channel mean and std are computed on the 45k training images only and applied to both splits.
The whole dataset is kept on the GPU as tensors, which is much faster than a `DataLoader` for 32×32 images.
The **test set is not loaded here**; it is used only once, in Task 4.

In [ ]:
raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
X_all = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)  # (50000, 3, 32, 32) in [0, 1]
Y_all = torch.tensor(raw.targets)
CLASSES = raw.classes

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]

MEAN = X_all[train_idx].mean(dim=(0, 2, 3))
STD = X_all[train_idx].std(dim=(0, 2, 3))
normalize = lambda x: (x - MEAN.view(1, 3, 1, 1)) / STD.view(1, 3, 1, 1)

X_train, Y_train = normalize(X_all[train_idx]).to(device), Y_all[train_idx].to(device)
X_val, Y_val = normalize(X_all[val_idx]).to(device), Y_all[val_idx].to(device)
del raw, X_all, Y_all

print('train', tuple(X_train.shape), '| val', tuple(X_val.shape))
print('mean', MEAN.numpy().round(4), '| std', STD.numpy().round(4))
print('val class counts', torch.bincount(Y_val.cpu()).tolist())

In [ ]:
def batches(X, Y, batch_size, shuffle=True):
    """Yield (x, y) mini-batches from tensors that already live on the device."""
    n = len(X)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield X[j], Y[j]

## Task 1: Backpropagation

### (a) Layers with explicit forward and backward passes

Each layer caches what it needs during `forward` and returns $\partial L/\partial(\text{input})$ from `backward`.
No autograd is used: all tensors are plain tensors with `requires_grad=False`.

For a mini-batch $X \in \mathbb{R}^{N\times d_{in}}$ and upstream gradient $G = \partial L/\partial Y$:

| Layer | Forward | Backward |
|---|---|---|
| Linear | $Y = XW^\top + b$ | $\frac{\partial L}{\partial W} = G^\top X,\quad \frac{\partial L}{\partial b} = \sum_n G_n,\quad \frac{\partial L}{\partial X} = G W$ |
| ReLU | $Y = \max(0, X)$ | $\frac{\partial L}{\partial X} = G \odot \mathbb{1}[X > 0]$ |
| Sigmoid | $Y = \sigma(X)$ | $\frac{\partial L}{\partial X} = G \odot Y \odot (1 - Y)$ |
| Softmax + cross-entropy | $p = \mathrm{softmax}(z),\; L = -\frac{1}{N}\sum_n \log p_{n,y_n}$ | $\frac{\partial L}{\partial z} = \frac{1}{N}(p - \mathrm{onehot}(y))$ |

$W$ is stored as (out, in), the same layout as `nn.Linear`, so weights can be copied across for the autograd check in 1(b).
Softmax subtracts the row maximum before exponentiating for numerical stability (this does not change the result).
Sigmoid is included here because it is needed in 1(d).

In [ ]:
class Linear:
    def __init__(self, in_features, out_features, init='kaiming', device=device, dtype=torch.float32):
        # kaiming: std = sqrt(2 / fan_in) (for ReLU); xavier: std = sqrt(1 / fan_in) (for sigmoid/tanh)
        gain = 2.0 if init == 'kaiming' else 1.0
        self.W = torch.randn(out_features, in_features, device=device, dtype=dtype) * math.sqrt(gain / in_features)
        self.b = torch.zeros(out_features, device=device, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, grad_out):
        self.dW = grad_out.T @ self.x
        self.db = grad_out.sum(dim=0)
        return grad_out @ self.W

    def params_and_grads(self):
        return [(self.W, self.dW), (self.b, self.db)]


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, grad_out):
        return grad_out * self.mask

    def params_and_grads(self):
        return []


class Sigmoid:
    def forward(self, x):
        self.y = torch.sigmoid(x)
        return self.y

    def backward(self, grad_out):
        return grad_out * self.y * (1 - self.y)

    def params_and_grads(self):
        return []


class SoftmaxCrossEntropy:
    """Mean softmax cross-entropy over the batch; takes raw logits and integer labels."""
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values
        log_p = z - z.exp().sum(dim=1, keepdim=True).log()
        self.p = log_p.exp()
        self.y = y
        return -log_p[torch.arange(len(y)), y].mean()

    def backward(self):
        grad = self.p.clone()
        grad[torch.arange(len(self.y)), self.y] -= 1
        return grad / len(self.y)


class MLP:
    """Stack of Linear layers with an activation between them (none after the last layer)."""
    def __init__(self, sizes, activation='relu', device=device, dtype=torch.float32):
        act_cls = {'relu': ReLU, 'sigmoid': Sigmoid}[activation]
        init = 'kaiming' if activation == 'relu' else 'xavier'
        self.layers = []
        for i, (d_in, d_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(d_in, d_out, init=init, device=device, dtype=dtype))
            if i < len(sizes) - 2:
                self.layers.append(act_cls())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        x = x.flatten(1)
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        grad = self.loss_fn.backward()
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def linear_layers(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def params_and_grads(self):
        return [pg for l in self.layers for pg in l.params_and_grads()]

Quick sanity check. At initialization the network has learned nothing, so the loss should be of the order of
$\ln 10 \approx 2.303$ (the loss of a uniform guess). It is somewhat higher because Kaiming initialization gives logits
with standard deviation $\approx 1.4$ rather than 0, so predictions are random but not uniform.
Each row of $\partial L/\partial z$ must sum to zero, because the softmax probabilities sum to one.

In [ ]:
set_seed()
mlp = MLP([3 * 32 * 32, 256, 128, 10])
xb, yb = X_train[:256], Y_train[:256]
loss = mlp.loss(xb, yb)
mlp.backward()

print(f'initial loss {loss.item():.4f}  (ln 10 = {math.log(10):.4f}) | logit std {mlp.forward(xb).std().item():.3f}')
print('logit-gradient row sums ~ 0:', mlp.loss_fn.backward().sum(dim=1).abs().max().item())
for i, l in enumerate(mlp.linear_layers()):
    print(f'Linear {i}: W {tuple(l.W.shape)}  dW {tuple(l.dW.shape)}  db {tuple(l.db.shape)}')
print('number of parameters:', sum(p.numel() for p, _ in mlp.params_and_grads()))